# Normalizing-flow proposal

`NormalizingFlowProposal` fits a copula normalizing flow (via the optional
[`coppuccino`](https://github.com/AaronDJohnson/coppuccino) package) to the
cold chain's recent samples and uses it as an **independence proposal**
for Metropolis-Hastings. Once the flow has learned the rough shape of
the posterior, draws from it have high acceptance even for targets where
AM/SCAM struggle (e.g. curved or banana-shaped distributions).

This notebook:
1. Sets up a curved 2-D target (banana).
2. Compares chains using standard adaptive proposals alone vs. with the NF
   proposal added.
3. Inspects acceptance, mixing, and recovered density.

The dependency is optional — if `coppuccino` is not installed, instantiating
the proposal raises `ImportError` with an install hint.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from impulse.samplers import PTSampler
from impulse.flow_proposals import NormalizingFlowProposal

rng = np.random.default_rng(0)

## 1. Define the banana target

A classic Rosenbrock-style banana posterior. The log-density is
$$
\log p(x_1, x_2) = -\tfrac{1}{2}\left(\tfrac{x_1^2}{a^2} + b\,(x_2 + x_1^2 - 1)^2\right)
$$
Standard AM proposals mix slowly here; the NF can learn the curvature.

In [ ]:
A, B = 1.0, 8.0

def lnlike(x):
    x1, x2 = x[0], x[1]
    return float(-0.5 * (x1**2 / A**2 + B * (x2 + x1**2 - 1.0)**2))

def lnprior(x):
    if np.any(np.abs(x) > 8):
        return -np.inf
    return 0.0

# Visualize the target on a grid
xx = np.linspace(-4, 4, 200)
yy = np.linspace(-6, 3, 200)
Xg, Yg = np.meshgrid(xx, yy)
Zg = np.array([[lnlike(np.array([x, y])) for x in xx] for y in yy])
plt.figure(figsize=(6, 5))
plt.contourf(Xg, Yg, np.exp(Zg - Zg.max()), levels=30, cmap='viridis')
plt.title('Target density (banana)')
plt.xlabel('x1'); plt.ylabel('x2')
plt.show()

## 2. Baseline run — standard adaptive proposals only

In [ ]:
baseline = PTSampler(
    ndim=2, lnlike=lnlike, lnprior=lnprior,
    am_weight=15, scam_weight=30, de_weight=50,
    ntemps=1, seed=1, outdir='./_chains_baseline',
    buffer_size=2000, cov_update=100, save_freq=2000,
)
baseline.sample(np.array([[0.5, 0.0]]), num_iterations=8000)
data_b = baseline.load_chain()
s_b = data_b['samples'][0]
print('baseline acceptance rates:', {k: v['rate'] for k, v in baseline.proposal_acceptance_rates().items()})

## 3. NF-augmented run

Add `NormalizingFlowProposal` alongside the standard proposals. The NF
needs samples to fit, so it sits idle (as a stay-put no-op) until
`min_samples` accepted samples are in the buffer. After that, it refits
every `refit_interval` cold-chain calls.

In [ ]:
nf = NormalizingFlowProposal(
    min_samples=1500,       # wait for the buffer to fill before fitting
    refit_interval=1500,    # refit every 1500 cold-chain proposals
    max_epochs=120,         # forwarded to coppuccino.normalizing_flows_fit
    prior_bounds=np.array([[-8.0, 8.0], [-8.0, 8.0]]),  # match the prior support
    rng_seed=2,
)

nf_sampler = PTSampler(
    ndim=2, lnlike=lnlike, lnprior=lnprior,
    am_weight=15, scam_weight=30, de_weight=50,
    ntemps=1, seed=2, outdir='./_chains_nf',
    buffer_size=2000, cov_update=100, save_freq=2000,
)
nf_sampler.proposal_bundle.add_jump(nf, weight=30.0)
nf_sampler.sample(np.array([[0.5, 0.0]]), num_iterations=8000)

data_n = nf_sampler.load_chain()
s_n = data_n['samples'][0]
rates = nf_sampler.proposal_acceptance_rates()
print('NF run acceptance rates:', {k: v['rate'] for k, v in rates.items()})
print(f'NF refits performed:    {nf._fit_count}')

## 4. Compare recovered densities

In [ ]:
burn = 2500
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
axes[0].contourf(Xg, Yg, np.exp(Zg - Zg.max()), levels=30, cmap='viridis')
axes[0].set_title('Truth')
axes[1].hexbin(s_b[burn:, 0], s_b[burn:, 1], gridsize=40, cmap='viridis', extent=(-4, 4, -6, 3))
axes[1].set_title('Baseline (AM+SCAM+DE)')
axes[2].hexbin(s_n[burn:, 0], s_n[burn:, 1], gridsize=40, cmap='viridis', extent=(-4, 4, -6, 3))
axes[2].set_title('+ NF proposal')
for ax in axes:
    ax.set_xlabel('x1')
axes[0].set_ylabel('x2')
plt.tight_layout(); plt.show()

## 5. Mixing diagnostics

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(s_b[:, 0], lw=0.5, label='baseline')
axes[0].plot(s_n[:, 0], lw=0.5, label='+ NF', alpha=0.7)
axes[0].set_ylabel('x1'); axes[0].legend(loc='upper right')
axes[1].plot(s_b[:, 1], lw=0.5, label='baseline')
axes[1].plot(s_n[:, 1], lw=0.5, label='+ NF', alpha=0.7)
axes[1].set_ylabel('x2'); axes[1].set_xlabel('iteration')
plt.tight_layout(); plt.show()

## 6. Fixed-flow mode (no refitting)

If you already have a fitted `coppuccino` flow — for example from a
previous run, importance-sampled draws, or simulator output — pass it
directly via `flow=...`. In this mode the proposal is active from
iteration 0 (no warm-up) and never refits.

In [ ]:
from coppuccino import normalizing_flows_fit

# Pretend this came from an earlier run on the same problem.
pretrained_flow = normalizing_flows_fit(
    s_n[2500:], max_epochs=120, rng_seed=11,
    prior_bounds=np.array([[-8.0, 8.0], [-8.0, 8.0]]),
)

fixed_nf = NormalizingFlowProposal(flow=pretrained_flow)
print('fixed mode:', fixed_nf.fixed)

fixed_sampler = PTSampler(
    ndim=2, lnlike=lnlike, lnprior=lnprior,
    am_weight=15, scam_weight=30, de_weight=50,
    ntemps=1, seed=3, outdir='./_chains_fixed_nf',
    buffer_size=2000, cov_update=100, save_freq=2000,
)
fixed_sampler.proposal_bundle.add_jump(fixed_nf, weight=50.0)
fixed_sampler.sample(np.array([[0.5, 0.0]]), num_iterations=4000)

print('refits performed (should be 0):', fixed_nf._fit_count)
print('rates:', {k: round(v['rate'], 3) for k, v in fixed_sampler.proposal_acceptance_rates().items()})

Flows can also be persisted to disk and reloaded later:

```python
from coppuccino import save_flow, load_flow
save_flow(pretrained_flow, 'banana_flow.pkl')
# ... later ...
flow = load_flow('banana_flow.pkl')
nf = NormalizingFlowProposal(flow=flow)
```

Because pickling drops the JAX flow object, restoring a fixed-flow
proposal from a sampler checkpoint requires re-attaching the flow:
`nf.set_flow(flow)`.

## Notes

- **`flow=...`**: pass a pre-fitted flow to skip warm-up and disable
  refitting entirely. Useful when you already have a density estimate.
- **`min_samples`**: in adaptive mode, samples to gather before the
  first flow fit. Until then, the proposal is a no-op (`qxy=0`); pair
  it with a standard proposal during warm-up.
- **`refit_interval`**: cost vs. adaptation. Each fit is seconds-scale.
- **`prior_bounds`**: strongly recommended when the prior is bounded;
  prevents the flow from extrapolating into unsupported regions.
- **PT**: the NF fits only on the cold chain (`cold_chain_only=True`);
  hot chains reuse the same flow. MH handles the proposal/target mismatch.
- **Checkpointing**: in adaptive mode the flow is dropped during pickle
  and refits on resume. In fixed-flow mode, re-attach the flow with
  `nf.set_flow(flow)` after restoring.
- **Optional dep**: if `coppuccino` is not installed, instantiating
  `NormalizingFlowProposal` raises `ImportError` with an install hint.